In [ ]:
import requests
import pandas as pd
from datetime import datetime
from dateutil import parser
import pytz
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets


def get_today_game_ids():
    url = "https://statsapi.mlb.com/api/v1/schedule?sportId=1"
    resp = requests.get(url).json()
    games = resp.get('dates', [])
    if not games:
        return []
    return [game['gamePk'] for game in games[0].get('games', [])]


def get_team_lineup(game_pk, team_type):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
    resp = requests.get(url).json()

    team_info = resp['liveData']['boxscore']['teams'][team_type]
    team_name = team_info['team']['name']

    lineup = []
    for pid, pdata in team_info['players'].items():
        bo = pdata.get('battingOrder')
        if bo:
            try:
                bo_int = int(bo)
                if 100 <= bo_int <= 900:
                    lineup.append({
                        'Order': bo_int // 100,
                        'Name': pdata['person']['fullName'],
                        'Pos': pdata['position']['abbreviation']
                    })
            except Exception:
                continue

    lineup = sorted(lineup, key=lambda x: x['Order'])
    df = pd.DataFrame(lineup)
    if df.empty:
        df = pd.DataFrame(columns=['Name', 'Pos'])

    # Get starting pitcher name
    sp_name = "-"
    try:
        sp_id = team_info.get('startingPitcher', None)
        if sp_id:
            sp_key = f'ID{sp_id}'
            if sp_key in team_info['players']:
                sp_name = team_info['players'][sp_key]['person']['fullName']
        else:
            pitchers = team_info.get('pitchers', [])
            if pitchers:
                sp_name = team_info['players'][f'ID{pitchers[0]}']['person']['fullName']
    except Exception:
        sp_name = "-"

    return team_name, df, sp_name


def get_venue_info(game_pk):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
    resp = requests.get(url).json()
    try:
        venue_name = resp['gameData']['venue']['name']
        venue_id = resp['gameData']['venue']['id']
    except Exception:
        venue_name = "-"
        venue_id = "-"
    return venue_name, venue_id


def get_game_start_time(game_pk):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
    resp = requests.get(url).json()
    try:
        game_date_str = resp['gameData']['datetime']['dateTime']  # ISO8601 with tz
        dt_utc = parser.isoparse(game_date_str)
        est = pytz.timezone('US/Eastern')
        dt_est = dt_utc.astimezone(est)
        return dt_est.strftime('%I:%M %p EST').lstrip('0')
    except Exception:
        return "-"


def show_all_lineups(_=None):
    clear_output(wait=True)
    now_str = datetime.now().strftime('%I:%M:%S %p').lstrip('0')
    print(f"Updated at {now_str}\n")

    game_ids = get_today_game_ids()
    if not game_ids:
        print("No games today.")
        return

    for game_pk in game_ids:
        try:
            away_name, away_df, away_sp = get_team_lineup(game_pk, 'away')
            home_name, home_df, home_sp = get_team_lineup(game_pk, 'home')
            venue_name, venue_id = get_venue_info(game_pk)
            game_time_str = get_game_start_time(game_pk)

            away_df.index = list(range(1, len(away_df) + 1))
            home_df.index = list(range(1, len(home_df) + 1))

            combined = pd.DataFrame(index=range(1, 10))
            combined['Name'] = away_df['Name']
            combined['Pos'] = away_df['Pos']
            combined['Name_2'] = home_df['Name']
            combined['Pos_2'] = home_df['Pos']

            sp_row = pd.DataFrame({
                'Name': [away_sp],
                'Pos': ['SP'],
                'Name_2': [home_sp],
                'Pos_2': ['SP']
            }, index=[0])

            combined = pd.concat([sp_row, combined])
            combined.fillna("-", inplace=True)

            # Rename columns to avoid .1 and keep clear separation for styling
            combined.columns = ['Name', 'Pos', 'Name_2', 'Pos_2']

            # Display header and venue/time centered
            display(HTML(f"<div style='text-align:center; font-weight:bold; font-size:16px; margin-bottom:5px;'>{away_name} @ {home_name}</div>"))
            display(HTML(f"<div style='text-align:center; font-style:italic; margin-bottom:5px;'>{venue_name} ({venue_id})</div>"))
            display(HTML(f"<div style='text-align:center; font-style:italic; margin-bottom:10px;'>{game_time_str}</div>"))

            styled = combined.style.set_table_attributes(
                "style='margin-left:auto; margin-right:auto; border-collapse: collapse;'"
            ).set_table_styles([
                {'selector': 'th', 'props': [('text-align', 'center')]},
                {'selector': 'td', 'props': [('text-align', 'center')]}
            ]).set_caption("Lineups and Starters")

            # Rename columns for display (no suffixes)
            combined.columns = ['Name', 'Pos', 'Name', 'Pos']

            display(styled)

        except Exception as e:
            print(f"Error loading game {game_pk}: {e}")


input_widget = widgets.Text(description="Press Enter to refresh:")
input_widget.continuous_update = False
input_widget.on_submit(show_all_lineups)

display(input_widget)

show_all_lineups()


In [ ]:
import requests
import pandas as pd
from datetime import datetime
from dateutil import parser
import pytz
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets


def get_today_game_ids():
    url = "https://statsapi.mlb.com/api/v1/schedule?sportId=1"
    resp = requests.get(url).json()
    games = resp.get('dates', [])
    if not games:
        return []
    return [game['gamePk'] for game in games[0].get('games', [])]


def get_team_lineup(game_pk, team_type):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
    resp = requests.get(url).json()

    team_info = resp['liveData']['boxscore']['teams'][team_type]
    team_name = team_info['team']['name']

    lineup = []
    for pid, pdata in team_info['players'].items():
        bo = pdata.get('battingOrder')
        if bo:
            try:
                bo_int = int(bo)
                if 100 <= bo_int <= 900:
                    lineup.append({
                        'Order': bo_int // 100,
                        'Name': pdata['person']['fullName'],
                        'Pos': pdata['position']['abbreviation']
                    })
            except Exception:
                continue

    lineup = sorted(lineup, key=lambda x: x['Order'])
    df = pd.DataFrame(lineup)
    if df.empty:
        df = pd.DataFrame(columns=['Name', 'Pos'])

    # Get starting pitcher name
    sp_name = "-"
    try:
        sp_id = team_info.get('startingPitcher', None)
        if sp_id:
            sp_key = f'ID{sp_id}'
            if sp_key in team_info['players']:
                sp_name = team_info['players'][sp_key]['person']['fullName']
        else:
            pitchers = team_info.get('pitchers', [])
            if pitchers:
                sp_name = team_info['players'][f'ID{pitchers[0]}']['person']['fullName']
    except Exception:
        sp_name = "-"

    return team_name, df, sp_name


def get_venue_info(game_pk):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
    resp = requests.get(url).json()
    try:
        venue_name = resp['gameData']['venue']['name']
        venue_id = resp['gameData']['venue']['id']
    except Exception:
        venue_name = "-"
        venue_id = "-"
    return venue_name, venue_id


def get_game_start_time(game_pk):
    url = f"https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
    resp = requests.get(url).json()
    try:
        game_date_str = resp['gameData']['datetime']['dateTime']  # ISO8601 with tz
        dt_utc = parser.isoparse(game_date_str)
        est = pytz.timezone('US/Eastern')
        dt_est = dt_utc.astimezone(est)
        return dt_est.strftime('%I:%M %p EST').lstrip('0')
    except Exception:
        return "-"


def show_all_lineups(_=None):
    clear_output(wait=True)
    now_str = datetime.now().strftime('%I:%M:%S %p').lstrip('0')
    print(f"Updated at {now_str}\n")

    game_ids = get_today_game_ids()
    if not game_ids:
        print("No games today.")
        return

    all_game_html = []  # To hold each game’s HTML block

    for game_pk in game_ids:
        try:
            away_name, away_df, away_sp = get_team_lineup(game_pk, 'away')
            home_name, home_df, home_sp = get_team_lineup(game_pk, 'home')
            venue_name, venue_id = get_venue_info(game_pk)
            game_time_str = get_game_start_time(game_pk)

            away_df.index = list(range(1, len(away_df) + 1))
            home_df.index = list(range(1, len(home_df) + 1))

            combined = pd.DataFrame(index=range(1, 10))
            combined['Name'] = away_df['Name']
            combined['Pos'] = away_df['Pos']
            combined['Name'] = combined['Name']  # to avoid pandas warning
            combined['Pos'] = combined['Pos']
            combined['Name_2'] = home_df['Name']
            combined['Pos_2'] = home_df['Pos']

            sp_row = pd.DataFrame({
                'Name': [away_sp],
                'Pos': ['SP'],
                'Name_2': [home_sp],
                'Pos_2': ['SP']
            }, index=[0])

            combined = pd.concat([sp_row, combined])
            combined.fillna("-", inplace=True)

            # Rename columns to remove suffixes for display
            combined.columns = ['Name', 'Pos', 'Name', 'Pos']

            # Create styled table and convert to HTML
            styled = combined.style.set_table_attributes(
                "style='margin-left:auto; margin-right:auto; border-collapse: collapse;'"
            ).set_table_styles([
                {'selector': 'th', 'props': [('text-align', 'center')]},
                {'selector': 'td', 'props': [('text-align', 'center')]}
            ]).set_caption("Lineups and Starters")

            table_html = styled.to_html()

            # Build the full block HTML for this game with header, venue, time, and table
            game_html = f"""
            <div style="display:inline-block; vertical-align:top; width:30%; margin-right: 20px; box-sizing: border-box;">
                <div style="text-align:center; font-weight:bold; font-size:16px; margin-bottom:5px;">{away_name} @ {home_name}</div>
                <div style="text-align:center; font-style:italic; margin-bottom:5px;">{venue_name} ({venue_id})</div>
                <div style="text-align:center; font-style:italic; margin-bottom:10px;">{game_time_str}</div>
                {table_html}
            </div>
            """
            all_game_html.append(game_html)

        except Exception as e:
            print(f"Error loading game {game_pk}: {e}")

    # Join all games blocks side-by-side
    display(HTML("<div style='width:100%; display:flex; flex-wrap: wrap; justify-content: center;'>" +
                 "".join(all_game_html) +
                 "</div>"))


input_widget = widgets.Text(description="Press Enter to refresh:")
input_widget.continuous_update = False
input_widget.on_submit(show_all_lineups)

display(input_widget)

show_all_lineups()
